# 01 — Data coverage and Gold/Silver correlations

This notebook loads the reproducible market snapshot, checks coverage, plots normalized prices and returns, and measures contemporaneous and lead/lag dependence. Correlation is descriptive and is not evidence of causality.

In [1]:
from pathlib import Path
import os
import sys
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = (ROOT / '..').resolve()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / 'src'))

from gold_silver.config import load_config
from gold_silver.data import load_cached_market_data
from gold_silver.features import build_features
from gold_silver.analysis import correlation_report, summarize_correlations

config = load_config(ROOT / 'configs/default.yaml')
raw = load_cached_market_data(config)
features = build_features(raw, config.features)
print(f'Raw rows: {len(raw):,}')
print(f'Feature rows: {len(features):,} | columns: {features.shape[1]}')
print(f'Date range: {features.index.min().date()} to {features.index.max().date()}')
features.head()

Raw rows: 26,290
Feature rows: 6,510 | columns: 148
Date range: 2000-08-30 to 2026-08-12


,gold_close,gold_return,gold_return_current,gold_ohlc_available,gold_intraday_return,gold_range_log,gold_close_location,gold_overnight_gap,gold_volume_log,gold_volume_change,...,sp500_close_change_lag_5,sp500_close_change_lag_20,tnx_close_change,tnx_close_change_lag_1,tnx_close_change_lag_5,tnx_close_change_lag_20,vix_close_change,vix_close_change_lag_1,vix_close_change_lag_5,vix_close_change_lag_20
Date,,,,,,,,,,,,,,,,,,,,,
2000-08-30,273.899994,NaN,NaN,1.0,0.000000,0.000000,0.5,0.000000,0.000000,NaN,...,0.005220,0.000417,-0.001378,0.007258,-0.006442,-0.003679,0.046278,0.020940,-0.005165,-0.027129
2000-08-31,278.299988,0.015937,0.015937,1.0,0.012656,0.012656,1.0,0.003280,0.000000,NaN,...,0.001553,0.009588,-0.012317,-0.001378,-0.001573,-0.004197,-0.049243,0.046278,-0.019756,-0.000500
2000-09-01,277.000000,-0.004682,-0.004682,1.0,0.000000,0.000000,0.5,-0.004682,0.000000,NaN,...,-0.001234,0.007114,-0.009470,-0.012317,0.000874,-0.007260,0.040157,-0.049243,-0.030387,-0.070996
2000-09-05,275.799988,-0.004342,-0.004342,1.0,0.000000,0.000000,0.5,-0.004342,1.098612,NaN,...,0.005059,0.011141,0.001409,-0.009470,0.007835,0.007933,0.122778,0.040157,0.000605,0.021780
2000-09-06,274.200012,-0.005818,-0.005818,1.0,0.000000,0.000000,0.5,-0.005818,0.000000,-1.0,...,-0.002811,0.002350,0.005090,0.001409,0.007258,-0.005733,0.047781,0.122778,0.020940,-0.012692


In [2]:
sns.set_theme(style='whitegrid', context='notebook')
fig, axes = plt.subplots(2, 2, figsize=(15, 10), constrained_layout=True)
normalized = features[['gold_close', 'silver_close']].div(features[['gold_close', 'silver_close']].iloc[0]).mul(100)
normalized.plot(ax=axes[0, 0], color=['#d49a00', '#8c8c8c'], title='Normalized Gold and Silver prices (start = 100)')
axes[0, 0].set_ylabel('Index')
features[['gold_return', 'silver_return']].plot(ax=axes[0, 1], alpha=0.7, color=['#d49a00', '#5d5d5d'], title='Daily log returns')
axes[0, 1].set_ylabel('Log return')
features['gold_return'].rolling(60).corr(features['silver_return']).plot(ax=axes[1, 0], color='#244a7c', title='60-day rolling return correlation')
axes[1, 0].axhline(0, color='black', linewidth=0.8)
features['gold_close'].div(features['silver_close']).plot(ax=axes[1, 1], color='#7a3e9d', title='Gold/Silver price ratio')
axes[1, 1].set_ylabel('Ratio')
plt.show()

/var/folders/ys/hvb2c_0n7lb1bsh31xyf6d2h0000gp/T/ipykernel_89239/3421564259.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [3]:
report = correlation_report(features)
display(report.round(4))
print(summarize_correlations(report))

fig, ax = plt.subplots(figsize=(10, 4))
lead_lag = report.query("analysis == 'lead_lag_returns'").set_index('window')['pearson']
lead_lag.plot(kind='bar', ax=ax, color=['#b33a3a' if x < 0 else '#2f6f4e' for x in lead_lag])
ax.set_title('Gold return correlation with shifted Silver returns')
ax.set_xlabel('Silver shift in days')
ax.set_ylabel('Pearson correlation')
ax.axhline(0, color='black', linewidth=0.8)
plt.show()

,analysis,window,pearson,spearman
0,levels,0,0.9257,0.9292
1,returns,0,0.7784,0.7676
2,rolling_returns,20,0.7824,0.8108
3,rolling_returns,60,0.7815,0.8039
4,rolling_returns,252,0.7801,0.7890
5,lead_lag_returns,-5,0.0038,0.0149
6,lead_lag_returns,-4,-0.0188,-0.0213
7,lead_lag_returns,-3,0.0167,0.0037
8,lead_lag_returns,-2,0.0102,0.0107
9,lead_lag_returns,-1,-0.0179,-0.0563


Price-level correlation: Pearson=0.926. Return correlation: Pearson=0.778, Spearman=0.768. Strongest observed lead/lag: lag 0 with Pearson=0.778. These statistics describe historical dependence; they do not establish causality or predictive power.


/var/folders/ys/hvb2c_0n7lb1bsh31xyf6d2h0000gp/T/ipykernel_89239/1040801857.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**How to read the plots.** Normalized prices start at 100 so the two metals can be compared despite different units. Daily log returns are percentage-like changes; the rolling correlation shows whether their co-movement is stable, while the lead/lag bars test timing association rather than causality.